<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page on one day (`report_date`) in two months (January ,February) 2026, keeping only days with `gsc_data_available IS TRUE`.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features**

From `fact_content_daily_performance`:

`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `ga4_pageviews`, `ga4_users`

From `dim_content`:

`content_created_date`, `keyword_hash_id`, `main_intent`, `backlinks`


**Label**

Percent change in average daily gsc_impressions per page between January–February 2026 and March–April 2026: (avg_period2 - avg_period1) / avg_period1 * 100.

`True` if below -20%, `False`  otherwise.

**Context**

`client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_start`, `ga4_data_start`.

**Excluded**

`fact_content_query_90d` table, because its fixed window (2 April to 30 June 2026) does not cover March.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")
SRC = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"


START, END = "2026-01-01", "2026-02-28"

df = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, gsc_data_available,
       gsc_impressions, gsc_clicks, gsc_sum_position,
       ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec, ga4_pageviews, ga4_users

FROM read_parquet('{SRC}', hive_partitioning=true)
WHERE month IN ('2026-01', '2026-02')
  AND report_date BETWEEN DATE '{START}' AND DATE '{END}'
  AND gsc_data_available IS TRUE
""").df()

df.head()


HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,ga4_pageviews,ga4_users
0,2026-01-01,client_62f4a7e64f5e0096,content_8c259838a3dfabe4,True,16,0,8,0,0,0,0,0
1,2026-01-01,client_62f4a7e64f5e0096,content_5bbd1b5ea9f4d15c,True,1,0,0,0,0,0,0,0
2,2026-01-01,client_62f4a7e64f5e0096,content_65b8a610174a1036,True,32,0,154,0,0,0,0,0
3,2026-01-01,client_62f4a7e64f5e0096,content_3fc85bdef381af2d,True,79,0,161,0,0,0,0,0
4,2026-01-01,client_62f4a7e64f5e0096,content_80071808216aef39,True,1041,0,5844,0,0,0,0,0


In [2]:
key = ["report_date", "client_hash_id", "content_hash_id"]
print("rows           :", len(df))
print("unique keys    :", len(df[key].drop_duplicates()))
print("duplicate rows :", df.duplicated(key).sum())
print("FROM -->", df["report_date"].min(),"     TO -->" , df["report_date"].max())
print("number of null values = ", df[["gsc_impressions", "ga4_sessions"]].isna().sum().to_dict())

rows           : 5018926
unique keys    : 5018926
duplicate rows : 0
FROM --> 2026-01-01 00:00:00      TO --> 2026-02-28 00:00:00
number of null values =  {'gsc_impressions': 0, 'ga4_sessions': 2646410}


### Building the features

All three features are knowable at the decision moment.

- `avg_impressions`: mean daily `gsc_impressions` over January.
- `avg_clicks`: mean daily `gsc_clicks` over January.
- `avg_ga4_users`: mean daily `ga4_users` over January, NULL when GA4 is missing.



To evaluate the model's  performance and analyze the impact of  data leakage, we split our two-month feature collection period into two distinct halves:
- **First Half (h1):** January 2026,
- **Second Half (h2):** February 2026, establish our target label(`y`).


In [6]:
import pandas as pd
import numpy as np

d = df.copy()
d["report_date"] = pd.to_datetime(d["report_date"])
d["half"] = np.where(d["report_date"].dt.month == 1, "h1", "h2")

keys = ["client_hash_id", "content_hash_id"]

h1 = d[d["half"] == "h1"]
pages = h1.groupby(keys).agg(
    avg_impressions=("gsc_impressions", "mean"),
    avg_clicks=("gsc_clicks", "mean"),
    avg_ga4_users=("ga4_users", "mean"),
).reset_index()

halves = d.pivot_table(index=keys, columns="half", values="gsc_impressions", aggfunc="mean").reset_index()
halves["y"] = (halves["h2"] - halves["h1"]) / halves["h1"].replace(0, np.nan) * 100

pages = pages.merge(halves[keys + ["y"]], on=keys, how="left")

pages = pages.dropna(subset=["y"]).copy()
pages["y"] = np.select([pages["y"] < -20, pages["y"] > 20], ["down", "up"], default="stable")
pages.head()






,client_hash_id,content_hash_id,avg_impressions,avg_clicks,avg_ga4_users,y
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,9.064516,0.096774,<NA>,stable
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,9.692308,0.000000,<NA>,down
3,client_0797ff3a1fc9a6a5,content_0b33d8960857ad90,1.647059,0.000000,<NA>,stable
4,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,3.700000,0.000000,<NA>,stable
5,client_0797ff3a1fc9a6a5,content_1207efddce873942,10.709677,0.096774,<NA>,up


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

def run(data, name="model"):
    feature_cols = [c for c in data.columns if c not in ["client_hash_id", "content_hash_id", "y"]]
    X = data[feature_cols].fillna(-1)
    y = data["y"]

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    print(f"[{name}] features: {feature_cols}")
    print(f"[{name}] baseline: {y_te.value_counts(normalize=True).max():.3f}")
    print(f"[{name}] accuracy: {accuracy_score(y_te, pred):.3f}")
    print(f"[{name}] macro F1: {f1_score(y_te, pred, average='macro'):.3f}")

run(pages, "clean")

[clean] features: ['avg_impressions', 'avg_clicks', 'avg_ga4_users']
[clean] baseline: 0.442
[clean] accuracy: 0.456
[clean] macro F1: 0.438


In [7]:
leak = d.groupby(keys)["gsc_impressions"].mean().reset_index(name="avg_impressions_both")
leaky = pages.merge(leak, on=keys, how="left")
run(leaky, "leaky")

[leaky] features: ['avg_impressions', 'avg_clicks', 'avg_ga4_users', 'avg_impressions_both']
[leaky] baseline: 0.467
[leaky] accuracy: 0.939
[leaky] macro F1: 0.931


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

at almost 50% of the rows the GA4 data is null

In [8]:
print("number of null values after the filter = ", df.loc[df["gsc_data_available"] == True, ["gsc_impressions", "ga4_sessions"]].isna().sum().to_dict())

number of null values after the filter =  {'gsc_impressions': 0, 'ga4_sessions': 2646410}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.